In [2]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

In [3]:
# import data
df = pd.read_csv('./Data/industry_vs_education.csv')

In [4]:
# clean data (remove irrelevant codes)
remove_codes = ['Supplementary Codes', 'Not stated', 'Not applicable', 'Total']
df = df[~df['1-digit level HEAP Level of Highest Educational Attainment'].isin(remove_codes)]

In [5]:
# assign relevant industries
awu_industries = [
    "Agriculture, Forestry and Fishing",
    "Mining",
    "Manufacturing",
    "Construction",
]

cfmeu_industries = [
    "Mining",
    "Manufacturing",
    "Construction",
]

etu_industries = [
    "Electricity, Gas, Water and Waste Services",
    "Construction",
    "Manufacturing",
    "Mining",
]

amwu_industries = [
    "Manufacturing",
    "Mining",
    "Electricity, Gas, Water and Waste Services",
]

rtbu_industries = [
    "Transport, Postal and Warehousing",
]

twu_industries = [
    "Transport, Postal and Warehousing",
]

sda_industries = [
    "Retail Trade",
    "Wholesale Trade",
]

uwu_industries = [
    "Accommodation and Food Services",
    "Agriculture, Forestry and Fishing",
    "Manufacturing",
    "Transport, Postal and Warehousing",
]

asu_industries = [
    "Information Media and Telecommunications",
    "Financial and Insurance Services",
    "Rental, Hiring and Real Estate Services",
    "Administrative and Support Services",
]

pa_industries = [
    "Information Media and Telecommunications",
    "Professional, Scientific and Technical Services",
    "Manufacturing",
    "Mining",
    "Electricity, Gas, Water and Waste Services",
]

cpsu_industries = [
    "Public Administration and Safety",
    "Information Media and Telecommunications",
]

anmf_industries = [
    "Health Care and Social Assistance",
]

fsu_industries = [
    "Financial and Insurance Services",
]

meaa_industries = [
    "Information Media and Telecommunications",
    "Arts and Recreation Services",
]


In [6]:
# create industry-relevant dataframes for each union
awu = df[df['1-digit level INDP Industry of Employment'].isin(awu_industries)]
cfmeu = df[df['1-digit level INDP Industry of Employment'].isin(cfmeu_industries)]
etu = df[df['1-digit level INDP Industry of Employment'].isin(etu_industries)]
amwu = df[df['1-digit level INDP Industry of Employment'].isin(amwu_industries)]
rtbu = df[df['1-digit level INDP Industry of Employment'].isin(rtbu_industries)]
twu = df[df['1-digit level INDP Industry of Employment'].isin(twu_industries)]
sda = df[df['1-digit level INDP Industry of Employment'].isin(sda_industries)]
uwu = df[df['1-digit level INDP Industry of Employment'].isin(uwu_industries)]
asu = df[df['1-digit level INDP Industry of Employment'].isin(asu_industries)]
pa = df[df['1-digit level INDP Industry of Employment'].isin(pa_industries)]
cpsu = df[df['1-digit level INDP Industry of Employment'].isin(cpsu_industries)]
anmf = df[df['1-digit level INDP Industry of Employment'].isin(anmf_industries)]
fsu = df[df['1-digit level INDP Industry of Employment'].isin(fsu_industries)]
meaa = df[df['1-digit level INDP Industry of Employment'].isin(meaa_industries)]

In [7]:
# calculate education distributions
union_dfs = {
    "awu": awu,
    "cfmeu": cfmeu,
    "etu": etu,
    "amwu": amwu,
    "rtbu": rtbu,
    "twu": twu,
    "sda": sda,
    "uwu": uwu,
    "asu": asu,
    "pa": pa,
    "cpsu": cpsu,
    "anmf": anmf,
    "fsu": fsu,
    "meaa": meaa
}

# initialise results dict
union_education = {}

for name, union_df in union_dfs.items():
    education = (
        union_df.groupby(
            "1-digit level HEAP Level of Highest Educational Attainment",
            as_index=False
        )["Count"]
        .sum()
    )

    # calculate percentage figures
    education["Count"] = (
        education["Count"] /
        education["Count"].sum()
        * 100
    )

    # create key-value pair for union
    union_education[name] = education

In [8]:
# visualise distributions
edu_dfs = list(union_education.values())

unions = [
    'AWU',
    'CFMEU',
    'ETU',
    'AMWU',
    'RTBU',
    'TWU',
    'SDA',
    'UWU',
    'ASU',
    'PA',
    'CPSU',
    'ANMF',
    'FSU',
    'MEAA'
]

# map to order X-axis values
education_order = [
    'Secondary Education - Years 9 and below',
    'Certificate I & II Level',
    'Secondary Education - Years 10 and above',
    'Certificate III & IV Level',
    'Advanced Diploma and Diploma Level',
    'Bachelor Degree Level',
    'Graduate Diploma and Graduate Certificate Level',
    'Postgraduate Degree Level'
]

for df in edu_dfs:
    for union in unions:
        col = '1-digit level HEAP Level of Highest Educational Attainment'

        # apply education_order mapping
        df[col] = pd.Categorical(
            df[col],
            categories=education_order,
            ordered=True
        )

        # sort values
        df = df.sort_values(col)

        plt.figure(figsize=(10, 5))

        # plot bar figure
        plt.bar(
            df[col],
            df['Count'],
            width=1.0
        )

        # apply titles and labels
        plt.title(f'Highest Completed Education for {union}-relevant Industries')
        plt.xlabel('Highest Completed Education')
        plt.ylabel('% of potential membership')

        plt.xticks(rotation=90)
        #plt.close() <--- uncomment to avoid automatically displaying visualisations in Jupyter Notebook

In [9]:
# map education rankings (for numeric analysis)
education_map = {
    'Postgraduate Degree Level': 8,
    'Graduate Diploma and Graduate Certificate Level': 7, 
    'Bachelor Degree Level': 6,
    'Advanced Diploma and Diploma Level': 5, 
    'Certificate III & IV Level': 4,
    'Secondary Education - Years 10 and above': 3,
    'Certificate I & II Level': 2,
    'Secondary Education - Years 9 and below': 1
}

# flipped map (for converting results back to human-readable)
flipped_education_map = {
    8: 'Postgraduate Degree Level',
    7: 'Graduate Diploma and Graduate Certificate Level',
    6: 'Bachelor Degree Level',
    5: 'Advanced Diploma and Diploma Level',
    4: 'Certificate III & IV Level',
    3: 'Secondary Education - Years 10 and above',
    2: 'Certificate I & II Level',
    1: 'Secondary Education - Years 9 and below'
}

# applying ranking mapping
for name, education_df in union_education.items():
    education_df['1-digit level HEAP Level of Highest Educational Attainment'] = (
        education_df['1-digit level HEAP Level of Highest Educational Attainment']
        .map(education_map)
    )

In [10]:
# get education summary statistics for each union
def weighted_median(values, weights):
    sorter = np.argsort(values)
    values = np.array(values)[sorter]
    weights = np.array(weights)[sorter]

    cumulative_weight = np.cumsum(weights)
    cutoff = weights.sum() / 2

    return values[cumulative_weight >= cutoff][0]


# initialise results array
summary_results = []

for union, df in zip(unions, edu_dfs):
    
    levels = df["1-digit level HEAP Level of Highest Educational Attainment"].astype(float)
    counts = df["Count"]

    # calculate weighted mean
    weighted_mean = np.average(levels, weights=counts)

    # create summary results dict
    summary = {
        "union": union,
        "mean": weighted_mean,
        "median": weighted_median(levels, counts),
        "std": np.sqrt(
            np.average(
                (levels - weighted_mean) ** 2,
                weights=counts
            )
        )
    }

    # append summary results dict to master summary results array
    summary_results.append(summary)

# convert summary results array to dataframe
summary_df = pd.DataFrame(summary_results)

# sort summary df by mean
summary_df = summary_df.sort_values(by=['mean'])

# create readable average levels (apply flipped education map)
summary_df['mean_level'] = (
    summary_df['mean']
    .round()
    .astype(int)
    .map(flipped_education_map)
)

summary_df['median_level'] = (
    summary_df['median']
    .astype(int)
    .map(flipped_education_map)
)

# write results df to csv
summary_df.to_csv('./Data/education_by_industry_results.csv')

In [11]:
# store summary statistics
summary_mean = summary_df['mean'].mean()
summary_median = summary_df['mean'].median()
summary_std_median = summary_df['std'].median()
summary_std_mean = summary_df['std'].mean()
std_max = summary_df['std'].max()
std_min = summary_df['std'].min()
summary_std_range = std_max - std_min

# print summary statistics
#print("Mean mean: " + str(summary_mean))
#print("Mean Median: " + str(summary_median))
#print("STD Median: " + str(summary_std_median))
#print("STD Mean: " + str(summary_std_mean))
#print("Min STD: " + str(std_min) + ", Max STD:" + str(std_max))
#print("STD range: " + str(summary_std_range))